In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from neuralprophet import NeuralProphet 
import torch
import time

## DIVISION: NON BOOKS

In [45]:
df = pd.read_parquet(r"D:\Continuum\dataset\kompasgramedia\data-full\df_full.parquet")
df['ID_STORE_PRODUCT'] = df['LOC'] + df['ITEMID']
df

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202204,E202205,E202206,E202207,E202208,E202209,E202210,E202211,E202212,ID_STORE_PRODUCT
0,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101021000126
1,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10103,GRAMEDIA BALI MAL GALERIA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,101031000126
2,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10104,GRAMEDIA BANDUNG MERDEKA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101041000126
3,gam,REGIONAL C,Jawa Tengah & DIY,YOGYA,10105,GRAMEDIA YOGYA SUDIRMAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101051000126
4,gam,REGIONAL C,Jawa Tengah & DIY,SEMARANG,10106,GRAMEDIA SEMARANG PANDANARAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101061000126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3499995,gam,REGIONAL C,Jawa Tengah & DIY,YOGYA,10127,GRAMEDIA YOGYA MAL MALIOBORO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101279902665
3499996,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10128,GRAMEDIA BANDUNG MAL PVJ,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101289902665
3499997,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10129,GRAMEDIA BANDUNG MAL TRANS STUDIO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101299902665
3499998,gam,REGIONAL A,Bali & Nusa Tenggara,SURABAYA,10130,GRAMEDIA KUPANG JEND. SUDIRMAN,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101309902665


In [46]:
season = pd.read_parquet(r"D:\Continuum\dataset\kompasgramedia\data-full\season-1-data.parquet")

df = df.merge(season[['ItemID', 'SEASON']], left_on='ITEMID', right_on='ItemID', how='left')
df.rename(columns={'SEASON': 'SEASON_MASTER'}, inplace=True)
df.drop(columns=['ItemID'], inplace=True)
df['SEASON_y'] = df['SEASON_y'].fillna('REGULER')
df['SEASON_y'].value_counts()

SEASON_y
REGULER                     2941674
HOLIDAY                      240416
TAB                          113112
TAB-HOLIDAY                  109791
RAMADHAN-HOLIDAY              44292
TAM-HOLIDAY                   31975
TAM-TAB-HOLIDAY               14774
TAM-TAB-HOLIDAY-RAMADHAN       3507
TAM                             373
RAMADHAN                         86
Name: count, dtype: int64

In [48]:
df_nonbooks = df[df['DIVISION'] == "DIV NON BOOKS"]
df_nonbooks

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202205,E202206,E202207,E202208,E202209,E202210,E202211,E202212,ID_STORE_PRODUCT,SEASON_y
0,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101021000126,REGULER
1,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10103,GRAMEDIA BALI MAL GALERIA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,101031000126,REGULER
2,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10104,GRAMEDIA BANDUNG MERDEKA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101041000126,REGULER
3,gam,REGIONAL C,Jawa Tengah & DIY,YOGYA,10105,GRAMEDIA YOGYA SUDIRMAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101051000126,REGULER
4,gam,REGIONAL C,Jawa Tengah & DIY,SEMARANG,10106,GRAMEDIA SEMARANG PANDANARAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101061000126,REGULER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3499995,gam,REGIONAL C,Jawa Tengah & DIY,YOGYA,10127,GRAMEDIA YOGYA MAL MALIOBORO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101279902665,REGULER
3499996,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10128,GRAMEDIA BANDUNG MAL PVJ,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101289902665,REGULER
3499997,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10129,GRAMEDIA BANDUNG MAL TRANS STUDIO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101299902665,REGULER
3499998,gam,REGIONAL A,Bali & Nusa Tenggara,SURABAYA,10130,GRAMEDIA KUPANG JEND. SUDIRMAN,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,101309902665,REGULER


## Konsisten Terjual 3 Tahun

In [49]:
df_nonbooks['TOTALSALES_2022'] = df_nonbooks.filter(like="R2022").sum(axis=1)
df_nonbooks['TOTALSALES_2023'] = df_nonbooks.filter(like="R2023").sum(axis=1)
df_nonbooks['TOTALSALES_2024'] = df_nonbooks.filter(like="R2024").sum(axis=1)

columns_to_keep = ['ID_STORE_PRODUCT', 'DIVISION', 'PARETO_FINAL'] + sorted(df_nonbooks.filter(like="R20").columns) + ['TOTALSALES_2022', 'TOTALSALES_2023', 'TOTALSALES_2024']
df_nonbooks[columns_to_keep].head()

WARNING - (py.warnings._showwarnmsg) - C:\Users\akmal\AppData\Local\Temp\ipykernel_24308\663358514.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nonbooks['TOTALSALES_2022'] = df_nonbooks.filter(like="R2022").sum(axis=1)

WARNING - (py.warnings._showwarnmsg) - C:\Users\akmal\AppData\Local\Temp\ipykernel_24308\663358514.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nonbooks['TOTALSALES_2023'] = df_nonbooks.filter(like="R2023").sum(axis=1)

WARNING - (py.warnings._showwarnmsg) - C:\Users\akma

,ID_STORE_PRODUCT,DIVISION,PARETO_FINAL,R202201,R202202,R202203,R202204,R202205,R202206,R202207,...,R202406,R202407,R202408,R202409,R202410,R202411,R202412,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024
0,101021000126,DIV NON BOOKS,P60,5.0,1.0,3.0,2.0,2.0,3.0,4.0,...,3.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,27.0,7.0
1,101031000126,DIV NON BOOKS,P60,14.0,2.0,4.0,0.0,18.0,5.0,1.0,...,2.0,10.0,10.0,0.0,0.0,0.0,0.0,155.0,159.0,54.0
2,101041000126,DIV NON BOOKS,P60,0.0,0.0,0.0,0.0,0.0,-1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,6.0,0.0
3,101051000126,DIV NON BOOKS,P60,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,101061000126,DIV NON BOOKS,P60,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0


In [50]:
# Seleksi kolom dengan prefix "TOTALSALES_20"
r20_columns = [col for col in df_nonbooks.columns if col.startswith('TOTALSALES_20')]

# Mengecek baris di mana semua kolom "TOTALSALES_20" memiliki nilai lebih dari 0
rows_with_all_r20_gt0 = df_nonbooks[r20_columns].gt(0).all(axis=1)

# Menghitung jumlah baris yang memenuhi kondisi
count = rows_with_all_r20_gt0.sum()

print(f"Jumlah item yang konsisten terjual selama 3 tahun > 0: {count} ({round(count/len(df_nonbooks)*100,2)}%)")

Jumlah item yang konsisten terjual selama 3 tahun > 0: 348882 (31.94%)


In [51]:
df_nonbooks = df_nonbooks[rows_with_all_r20_gt0]
df_nonbooks

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202208,E202209,E202210,E202211,E202212,ID_STORE_PRODUCT,SEASON_y,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024
0,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101021000126,REGULER,38.0,27.0,7.0
1,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10103,GRAMEDIA BALI MAL GALERIA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,2.0,0.0,0.0,101031000126,REGULER,155.0,159.0,54.0
6,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 1,10108,GRAMEDIA JKT MATRAMAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101081000126,REGULER,114.0,93.0,69.0
7,gam,REGIONAL F,Sumatera,SUMATERA 1,10109,GRAMEDIA MEDAN GAJAH MADA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101091000126,REGULER,122.0,105.0,56.0
8,gam,REGIONAL C,Jawa Tengah & DIY,SEMARANG,10110,GRAMEDIA SEMARANG SETIABUDI,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101101000126,REGULER,26.0,21.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3499990,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10120,GRAMEDIA BALI DUTA PLAZA,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,1.0,0.0,1.0,0.0,101209902665,REGULER,2.0,5.0,2.0
3499993,gam,REGIONAL E,Jabodetabek & Banten,JAKARTA 3,10124,GRAMEDIA TANGERANG PLAZA BINTARO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,101249902665,REGULER,5.0,7.0,3.0
3499995,gam,REGIONAL C,Jawa Tengah & DIY,YOGYA,10127,GRAMEDIA YOGYA MAL MALIOBORO,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,101279902665,REGULER,3.0,3.0,1.0
3499996,gam,REGIONAL F,Jawa Barat,JAWA BARAT,10128,GRAMEDIA BANDUNG MAL PVJ,9902665,ESTUDEE BINDER NOTE PP A5 BLUE,ESTUDEE,CONSGMN_TR,...,0.0,0.0,0.0,0.0,0.0,101289902665,REGULER,13.0,14.0,8.0


In [52]:
df_nonbooks['SEASON_y'].value_counts()

SEASON_y
REGULER                     291474
TAB                          43560
RAMADHAN-HOLIDAY             10270
TAB-HOLIDAY                   1566
TAM-TAB-HOLIDAY               1049
TAM-TAB-HOLIDAY-RAMADHAN       903
TAM                             60
Name: count, dtype: int64

In [53]:
df_nonbooks = df_nonbooks.head(100)
# df_nonbooks

## Loop Forecast

In [42]:
sub_tables = []
store_product = []

# Mengelompokkan baris dari DataFrame df berdasarkan nilai pada kolom ID_STORE_PRODUCT
for value, group in df_nonbooks.groupby('ID_STORE_PRODUCT'):
    sub_tables.append(group)
    store_product.append(value)

# Menampilkan jumlah sub-tabel yang telah dibuat
num_sub_tables = len(sub_tables)
print(f"Jumlah sub-tabel yang telah dibuat: {num_sub_tables}")
print(f"List nilai ID_STORE_PRODUCT yang sesuai: {store_product}")

Jumlah sub-tabel yang telah dibuat: 100
List nilai ID_STORE_PRODUCT yang sesuai: ['10100206006148', '10101206006148', '10102206006148', '10103206006148', '10104206006148', '10105206006148', '10106206006148', '10108206006148', '10109206006148', '10110206006148', '10111206006148', '10112206006148', '10113206006148', '10114206006148', '10115206006148', '10116206006148', '10117206006148', '10118206006148', '10119206006148', '10120206006148', '10123206006148', '10124206006148', '10125206006148', '10126206006148', '10127206006148', '10128206006148', '10129206006148', '10130206006148', '10134206006148', '10137206006148', '10138206006148', '10139206006148', '10140206006148', '10141206006148', '10142206006148', '10144206006148', '10146206006148', '10147206006148', '10148206006148', '10149206006148', '10150206006148', '10152206006148', '10153205970348', '10153206006148', '10155206006148', '10156206006148', '10159206006148', '10161206006148', '10163206006148', '10165206006148', '10166206006148', 

In [43]:
# Pendefinisian efek seasonal
df_holiday = pd.DataFrame(
    {
        "event": "holiday",
        "ds": pd.to_datetime(
            [
                "2022-11",
                "2022-12",
                "2023-01",                    
                "2023-11",
                "2023-12",
                "2024-01",
                "2024-12"
            ]            
        ),
    }
)

df_tam = pd.DataFrame(
    {
        "event": "tam",
        "ds": pd.to_datetime(
            [
                "2022-09",
                "2022-10",
                "2023-09",                    
                "2023-10",
                "2024-09",
                "2024-10"
            ]            
        ),
    }
)

df_tab = pd.DataFrame(
    {
        "event": "tab",
        "ds": pd.to_datetime(
            [
                "2022-07",
                "2022-08",
                "2023-07",                    
                "2023-08",
                "2024-07",
                "2024-08"
            ]            
        ),
    }
)

df_ramadhan = pd.DataFrame(
    {
        "event": "ramadhan",
        "ds": pd.to_datetime(
            [
                "2022-04",
                "2022-05",
                "2023-03",                    
                "2023-04",
                "2024-03",
                "2024-04"
            ]            
        ),
    }
)

In [44]:
final_tables = []  # List untuk menyimpan hasil forecast setiap sub_table
forecast_period = 4
cutoff_date = '2024-08-01'  # Tanggal batas data yang akan digunakan untuk forecast

# Proses untuk setiap sub-table (hanya indeks 0 dan 1)
for index in range(0, num_sub_tables): # Mengubah rentang untuk 2 sub-tabel pertama
#     print(f"Ongoing loop for sub_tables[{index}]")
    sample_data = sub_tables[index]
    sample_data_r = sample_data.filter(like='R20')

    # Mengubah kolom ke format YYYY-MM
    new_row = [f"{col[1:][:4]}-{col[1:][4:]}" for col in sample_data_r.columns]
    
    # Mengonversi new_row menjadi DataFrame dengan satu baris
    new_row_df = pd.DataFrame([new_row], columns=sample_data_r.columns)
    
    # Menggabungkan new_row_df dengan sample_data_r
    sample_data_r = pd.concat([new_row_df, sample_data_r], ignore_index=True).T
    sample_data_r.columns = ['ds', 'y']
    sample_data_r = sample_data_r.assign(
        ds=pd.to_datetime(sample_data_r['ds'], format='%Y-%m'),
        y=sample_data_r['y'].astype(float)
    )
    
    # Pendefinisian filtered_data berdasarkan cutoff_date
    if cutoff_date is not None:
        filtered_data = sample_data_r[sample_data_r['ds'] <= cutoff_date]
    else:
        filtered_data = sample_data_r

    # Mengatur model NeuralProphet
    torch.manual_seed(0)
    m = NeuralProphet(
        trend_global_local="global",
        season_global_local="local",
        changepoints_range=0.8,
        epochs=20,
        trend_reg=5,
        learning_rate=0.05
    )
    m.set_plotting_backend("plotly-static")
    
    # Menambahkan efek seasonal berdasarkan kategori SEASON_y
    season_category = sample_data['SEASON_y'].iloc[0] 
    df_season = pd.DataFrame()

    if "HOLIDAY" in season_category:
        m.add_events("holiday")
        df_season = pd.concat([df_season, df_holiday], ignore_index=True)

    if "TAM" in season_category:
        m.add_events("tam")
        df_season = pd.concat([df_season, df_tam], ignore_index=True)

    if "TAB" in season_category:
        m.add_events("tab")
        df_season = pd.concat([df_season, df_tab], ignore_index=True)

    if "RAMADHAN" in season_category:
        m.add_events("ramadhan")
        df_season = pd.concat([df_season, df_ramadhan], ignore_index=True)
        
#     print(f"Seasonal category {index}: {season_category}")
    
    # Hanya tambahkan efek jika bukan REGULER
    if not df_season.empty:
        filtered_data = m.create_df_with_events(filtered_data, df_season) 

    # Memisahkan data pelatihan dan validasi
    df_train, df_val = m.split_df(filtered_data, freq='M', valid_p=0.25)
    metrics = m.fit(df_train, freq='M', validation_df=df_val)

    # Melakukan prediksi
    future = m.make_future_dataframe(filtered_data, periods=forecast_period, n_historic_predictions=len(sample_data_r))
    forecast = m.predict(future)
#     print(f"{forecast}\n")

    # Mengambil hasil prediksi
    forecast_row = forecast[['ds', 'yhat1']].tail(forecast_period) 
    forecast_row.columns = ['ds', 'y']

    # Menggabungkan data asli dan hasil prediksi
    filtered_data = pd.concat([filtered_data, forecast_row], ignore_index=True)
    filtered_data['y'] = filtered_data['y'].apply(lambda x: max(0, np.round(x)))
    filtered_data['ds'] = filtered_data['ds'].dt.strftime('%Y-%m')
    
    # Mengubah format ds untuk menggantikan data asli
    filtered_data['ds'] = filtered_data['ds'].str.replace('-', '').astype(object)
#     print(f"{filtered_data}\n")
    filtered_data = filtered_data[['ds', 'y']]
    filtered_data = filtered_data.T
    filtered_data.columns = 'R' + filtered_data.loc['ds']
    filtered_data = filtered_data.drop('ds')    

    # Mengubah tipe data
    filtered_data = filtered_data.astype(float)
    filtered_data.index = sample_data.index

    # Mengidentifikasi kolom terakhir dari filtered_data sesuai forecast_period
    forecast_columns = filtered_data.columns[-forecast_period:]

    # Overwrite kolom-kolom terakhir dari sample_data dengan filtered_data
    filtered_data_prioritized = sample_data.copy()
    filtered_data_prioritized[forecast_columns] = filtered_data[forecast_columns]

    # Menggabungkan data dengan memprioritaskan sample_data kecuali kolom forecast_columns
    combined = filtered_data_prioritized.combine_first(sample_data)

    # Menambahkan kolom dari sample_data_r yang tidak ada di sample_data
    for col in filtered_data.columns: 
        if col not in combined.columns:
            combined[col] = filtered_data[col]

    # Mengatur urutan kolom sesuai sample_data
    ordered_columns = list(sample_data.columns) + [col for col in filtered_data.columns if col not in sample_data.columns]
    combined = combined[ordered_columns]

    # Mengupdate sub_tables
    sub_tables[index] = combined
    final_tables.append(combined)  # Menambahkan hasil forecast ke final_tables

# Menggabungkan semua sub_tables menjadi satu DataFrame
df_nonbooks_new = pd.concat(final_tables, ignore_index=True)

INFO - (NP.config.__post_init__) - Note: Trend changepoint regularization is experimental.
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [93.75]% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - MS
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [91.667]% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - MS
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disab

Ongoing loop for sub_tables[0]
Seasonal category 0: TAM-TAB-HOLIDAY-RAMADHAN


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency MS corresponds to [93.75]% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - MS
WARNING - (NP.data.splitting._make_future_dataframe) - Insufficient data for 36 historic forecasts, reduced to 32.
WARNING - (NP.data.splitting._make_future_dataframe) - Future values not supplied for user specified events. All events being treated as not occurring in future
WARNING - (py.warnings._showwarnmsg) - C:\Users\akmal\anaconda3\Lib\site-packages\neuralprophet\data\split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP

Predicting: 3it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


           ds      y       yhat1       trend  season_yearly  events_additive  \
0  2023-01-01   36.0   32.887291   38.012714     -27.974062        22.848639   
1  2023-02-01   54.0   22.536020   39.802299     -17.266277         0.000000   
2  2023-03-01   66.0   -5.188731   42.311546       7.032849       -54.533123   
3  2023-04-01   53.0   23.636263   46.313797      31.855591       -54.533123   
4  2023-05-01   59.0   30.377720   50.349224     -19.971502         0.000000   
5  2023-06-01   62.0   11.306664   55.372490     -44.065826         0.000000   
6  2023-07-01  123.0  116.738213   60.233715     145.179169       -88.674675   
7  2023-08-01   81.0   43.176353   65.427139      66.423882       -88.674675   
8  2023-09-01   56.0   54.179237   70.636490      -5.112938       -11.344315   
9  2023-10-01   47.0   50.940548   74.894157     -12.609295       -11.344315   
10 2023-11-01   61.0   41.428104   78.933533     -60.354065        22.848639   
11 2023-12-01   51.0   35.035931   82.86

In [37]:
df_nonbooks_new #model dengan event holiday bawaan neuralprophet (test 0.2)

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202212,ID_STORE_PRODUCT,SEASON_y,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024,R202501,R202502,R202503,R202504
0,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 3,10101,GRAMEDIA JKT PINTU AIR,1000128,3M POST-IT NOTE 657,3M,CREDIT_TR,...,0.0,101011000128,REGULER,22.0,19.0,9.0,2.0,2.0,2.0,2.0
1,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101021000126,REGULER,38.0,27.0,7.0,2.0,2.0,3.0,3.0
2,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10103,GRAMEDIA BALI MAL GALERIA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101031000126,REGULER,155.0,159.0,54.0,12.0,11.0,16.0,14.0
3,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 1,10108,GRAMEDIA JKT MATRAMAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101081000126,REGULER,114.0,93.0,69.0,11.0,10.0,15.0,14.0
4,gam,REGIONAL F,Sumatera,SUMATERA 1,10109,GRAMEDIA MEDAN GAJAH MADA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101091000126,REGULER,122.0,105.0,56.0,10.0,10.0,14.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,gam,REGIONAL G,Jawa Timur,SURABAYA,81118,GS GENTENG BANYUWANGI,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,811181000126,REGULER,9.0,10.0,5.0,1.0,1.0,2.0,1.0
96,gam,REGIONAL G,Jawa Timur,SURABAYA,81183,GS EXPO MOJOKERTO,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,811831000126,REGULER,16.0,17.0,3.0,1.0,1.0,3.0,1.0
97,bkm,REGIONAL C,Jawa Tengah & DIY,YOGYA,90101,GRAMEDIA CILACAP,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,901011000126,REGULER,26.0,20.0,3.0,1.0,2.0,2.0,2.0
98,bkm,REGIONAL F,Jawa Barat,JAWA BARAT,90103,GRAMEDIA GARUT,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,901031000126,REGULER,6.0,22.0,8.0,2.0,2.0,2.0,2.0


In [43]:
df_nonbooks_new #model sendiri, default (test 0.25)

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202212,ID_STORE_PRODUCT,SEASON_y,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024,R202501,R202502,R202503,R202504
0,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 3,10101,GRAMEDIA JKT PINTU AIR,1000128,3M POST-IT NOTE 657,3M,CREDIT_TR,...,0.0,101011000128,REGULER,22.0,19.0,9.0,0.0,0.0,0.0,0.0
1,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101021000126,REGULER,38.0,27.0,7.0,0.0,0.0,1.0,0.0
2,gam,REGIONAL G,Bali & Nusa Tenggara,BALI,10103,GRAMEDIA BALI MAL GALERIA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101031000126,REGULER,155.0,159.0,54.0,13.0,0.0,0.0,0.0
3,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 1,10108,GRAMEDIA JKT MATRAMAN,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101081000126,REGULER,114.0,93.0,69.0,14.0,0.0,5.0,4.0
4,gam,REGIONAL F,Sumatera,SUMATERA 1,10109,GRAMEDIA MEDAN GAJAH MADA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,101091000126,REGULER,122.0,105.0,56.0,10.0,2.0,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,gam,REGIONAL G,Jawa Timur,SURABAYA,81118,GS GENTENG BANYUWANGI,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,811181000126,REGULER,9.0,10.0,5.0,1.0,0.0,1.0,1.0
96,gam,REGIONAL G,Jawa Timur,SURABAYA,81183,GS EXPO MOJOKERTO,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,811831000126,REGULER,16.0,17.0,3.0,0.0,0.0,2.0,0.0
97,bkm,REGIONAL C,Jawa Tengah & DIY,YOGYA,90101,GRAMEDIA CILACAP,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,901011000126,REGULER,26.0,20.0,3.0,0.0,0.0,1.0,0.0
98,bkm,REGIONAL F,Jawa Barat,JAWA BARAT,90103,GRAMEDIA GARUT,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,901031000126,REGULER,6.0,22.0,8.0,1.0,1.0,0.0,0.0


In [66]:
df_nonbooks_new #model sendiri, default (test 0.25), data terakhir 2024-08

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202208,E202209,E202210,E202211,E202212,ID_STORE_PRODUCT,SEASON_y,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024
0,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 3,10101,GRAMEDIA JKT PINTU AIR,1000128,3M POST-IT NOTE 657,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101011000128,REGULER,22.0,19.0,9.0
1,gam,REGIONAL G,Jawa Timur,SURABAYA,10102,GRAMEDIA SURABAYA ROYAL PLAZA,1000126,3M POST-IT NOTE 656,3M,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,101021000126,REGULER,38.0,27.0,7.0


In [39]:
df_nonbooks_new #model penambahan efek seasonal

,dataareaid,REGIONAL,AREA,CLUSTER,LOC,STORE_NAME,ITEMID,ITEMDESC,BRAND,MODELGROUPID,...,E202208,E202209,E202210,E202211,E202212,ID_STORE_PRODUCT,SEASON_y,TOTALSALES_2022,TOTALSALES_2023,TOTALSALES_2024
0,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 3,10100,GRAMEDIA JKT GAJAH MADA,206006148,GRAMEDIA TAS GO GREEN MODERN,GRAMEDIA,CREDIT_TR,...,0.0,0.0,0.0,0.0,0.0,10100206006148,TAM-TAB-HOLIDAY-RAMADHAN,651.0,450.0,186.0
1,gam,REGIONAL B,Jabodetabek & Banten,JAKARTA 3,10101,GRAMEDIA JKT PINTU AIR,206006148,GRAMEDIA TAS GO GREEN MODERN,GRAMEDIA,CREDIT_TR,...,0.0,0.0,0.0,5.0,0.0,10101206006148,TAM-TAB-HOLIDAY-RAMADHAN,814.0,749.0,551.0
